# 01. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 02. Config

In [ ]:
CONFIG = {
    "csv_path": "/content/drive/MyDrive/Colab Notebooks/labels_synthetic_calibrated_with_path.csv",
    "base_dir": "/content/drive/MyDrive/Colab Notebooks/",
    "model_path": "/content/drive/MyDrive/Colab Notebooks/Saved Models/best_models/best_model.pth",
    "batch_size": 32,
    "img_size": 224,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

# 03. Model

In [ ]:
class CBMModel(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.features = nn.Sequential(*list(backbone.children())[:-1])

        self.shared = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.concept_head = nn.Linear(128, 4)   # NO, NC, CO, PSC
        self.presence_head = nn.Linear(128, 1)  # Cataract or not

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        shared = self.shared(x)
        concepts = self.concept_head(shared)
        presence = torch.sigmoid(self.presence_head(shared))
        return concepts, presence

# 04. Dataset

In [ ]:
class CataractDataset(Dataset):
    def __init__(self, df, transform, base_dir):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.base_dir = base_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        rel_path = row["relative_path"].replace("\\", "/")
        img_path = os.path.join(self.base_dir, rel_path)

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        labels = np.array([
            row["NO_pseudo"],
            row["NC_pseudo"],
            row["CO_pseudo"],
            row["PSC_pseudo"]
        ])  # NOT normalized

        return image, labels

# 05. Transform

In [ ]:
transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Severity
def get_severity_label(score):
    if score < 1.0: return "Normal"
    elif score < 2.0: return "Mild"
    elif score < 3.0: return "Moderate"
    elif score < 4.0: return "Marked"
    else: return "Severe"

def to_severity_vector(values):
    return [get_severity_label(v) for v in values]

# 06. Main Function

In [ ]:
def main():
    print("Loading dataset...")
    df = pd.read_csv(CONFIG["csv_path"])

    _, val_df = train_test_split(df, test_size=0.2, random_state=42)

    val_ds = CataractDataset(val_df, transform, CONFIG["base_dir"])
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False)

    print("Loading model...")
    model = CBMModel().to(CONFIG["device"])
    checkpoint = torch.load(CONFIG["model_path"], map_location=CONFIG["device"], weights_only=False)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    all_preds = []
    all_labels = []

    print("Running evaluation...")

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(CONFIG["device"])

            concepts, presence = model(images)
            preds = concepts.cpu().numpy()
            preds = np.clip(preds, 0, 1) * 5.0

            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Scatter Plot (Pred vs True)
    concept_names = ["NO", "NC", "CO", "PSC"]

    for i, name in enumerate(concept_names):
        plt.figure()
        plt.scatter(all_labels[:, i], all_preds[:, i])
        plt.xlabel("True Values")
        plt.ylabel("Predicted Values")
        plt.title(f"{name} - Prediction vs Ground Truth")
        plt.grid()
        plt.show()


    # Error Distribution
    errors = all_preds - all_labels

    for i, name in enumerate(concept_names):
        plt.figure()
        plt.hist(errors[:, i], bins=30)
        plt.title(f"{name} - Error Distribution")
        plt.xlabel("Prediction Error")
        plt.ylabel("Frequency")
        plt.grid()
        plt.show()


    # Per-Concept MAE & MSE
    print("\n===== Regression Metrics =====")

    for i, name in enumerate(concept_names):
        mae = mean_absolute_error(all_labels[:, i], all_preds[:, i])
        mse = mean_squared_error(all_labels[:, i], all_preds[:, i])
        
        print(f"{name}: MAE={mae:.4f}, MSE={mse:.4f}")


    # Worst Predictions (Top Errors)
    total_error = np.mean(np.abs(errors), axis=1)
    worst_indices = np.argsort(-total_error)[:5]

    print("\n===== Worst Predictions =====")
    for idx in worst_indices:
        print(f"\nSample {idx}")
        print("True:", all_labels[idx])
        print("Pred:", all_preds[idx])
        print("Error:", errors[idx])

    # Convert to severity
    pred_severity = np.array([to_severity_vector(p) for p in all_preds])
    true_severity = np.array([to_severity_vector(t) for t in all_labels])

    concept_names = [
        "Nuclear Opalescence (NO)",
        "Nuclear Color (NC)",
        "Cortical Opacity (CO)",
        "Posterior Subcapsular (PSC)"
    ]

    for i, name in enumerate(concept_names):
        print(f"\n===== {name} =====")
        print(classification_report(true_severity[:, i], pred_severity[:, i]))
        print("Confusion Matrix:")
        print(confusion_matrix(true_severity[:, i], pred_severity[:, i]))

if __name__ == "__main__":
    main()